In [ ]:
import pandas as pd, numpy as np, arviz as az, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

trace      = az.from_netcdf("outputs/posteriors/hbrm_trace.nc")
df_test    = pd.read_csv("data/processed/markets_test.csv")
categories = sorted(df_test["category_str"].unique().tolist())
n_cats     = len(categories)

In [ ]:
alpha_samples = trace.posterior["alpha"].values.reshape(-1, n_cats)
beta_samples  = trace.posterior["beta"].values.reshape(-1, n_cats)

cat_stats = []
for i, cat in enumerate(categories):
    a_samps = alpha_samples[:, i]
    b_samps = beta_samples[:, i]
    a_hdi   = az.hdi(a_samps, hdi_prob=0.94)
    b_hdi   = az.hdi(b_samps, hdi_prob=0.94)
    n_mkt   = (df_test["category_str"] == cat).sum()
    cat_stats.append({
        "category":       cat,
        "n_markets":      n_mkt,
        "alpha_mean":     a_samps.mean(),
        "alpha_std":      a_samps.std(),
        "alpha_hdi_lo":   a_hdi[0],
        "alpha_hdi_hi":   a_hdi[1],
        "alpha_sig":      (a_hdi[0] > 0) or (a_hdi[1] < 0),
        "beta_mean":      b_samps.mean(),
        "beta_std":       b_samps.std(),
        "beta_hdi_lo":    b_hdi[0],
        "beta_hdi_hi":    b_hdi[1],
        "beta_sig":       (b_hdi[0] > 1) or (b_hdi[1] < 1),
        "prob_overconf":  (b_samps < 1).mean(),
        "prob_underconf": (b_samps > 1).mean(),
    })

cat_df = pd.DataFrame(cat_stats)
print(cat_df[["category","n_markets","alpha_mean","alpha_sig","beta_mean","beta_sig",
              "prob_overconf"]].to_string(index=False))
cat_df.to_csv("outputs/per_category_posterior.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

ax.axhspan(0, ax.get_ylim()[1] if ax.get_ylim()[1] > 1.5 else 1.5, xmin=0.5,
           alpha=0.04, color='blue')
ax.axhspan(0, 1.5, xmin=0, xmax=0.5, alpha=0.04, color='orange')

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label="beta=1 (perfect slope)")
ax.axvline(0.0, color='gray', linestyle=':',  linewidth=1.5, alpha=0.7, label="alpha=0 (no bias)")

colors_scatter = plt.cm.RdBu(cat_df["prob_overconf"])

for _, row in cat_df.iterrows():
    ax.errorbar(
        row["alpha_mean"], row["beta_mean"],
        xerr=[[row["alpha_mean"] - row["alpha_hdi_lo"]],
              [row["alpha_hdi_hi"] - row["alpha_mean"]]],
        yerr=[[row["beta_mean"] - row["beta_hdi_lo"]],
              [row["beta_hdi_hi"] - row["beta_mean"]]],
        fmt='o', capsize=5, markersize=max(6, np.sqrt(row["n_markets"])*0.3),
        color='crimson' if row["prob_overconf"] > 0.7 else 'steelblue',
        alpha=0.85, linewidth=1.5
    )
    ax.annotate(row["category"],
                (row["alpha_mean"], row["beta_mean"]),
                textcoords="offset points", xytext=(10, 2),
                fontsize=9, fontweight='bold' if row["alpha_sig"] or row["beta_sig"] else 'normal')

ax.text( 0.05, 1.55, "Underconfident\n+ overestimates YES", fontsize=8, color='navy',  alpha=0.5, ha='left')
ax.text(-0.35, 1.55, "Underconfident\n+ underestimates YES", fontsize=8, color='navy', alpha=0.5, ha='left')
ax.text( 0.05, 0.45, "Overconfident\n+ overestimates YES",  fontsize=8, color='darkred',alpha=0.5, ha='left')
ax.text(-0.35, 0.45, "Overconfident\n+ underestimates YES", fontsize=8, color='darkred',alpha=0.5, ha='left')

ax.set_xlabel("alpha (Intercept) - Systematic Bias Direction\n(+) = underestimates YES, (-) = overestimates YES", fontsize=11)
ax.set_ylabel("beta (Slope) - Confidence Level\n(<1) = overconfident, (>1) = underconfident", fontsize=11)
ax.set_title("Per-Category Calibration Parameters (Posterior Mean +/- 94% HDI)\nMarker size proportional to number of test markets", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("outputs/figures/17_category_alpha_beta_scatter.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
cat_sorted = cat_df.sort_values("prob_overconf", ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, n_cats * 0.45)))
colors_bar = ['crimson' if p > 0.7 else ('steelblue' if p < 0.3 else 'gray')
              for p in cat_sorted["prob_overconf"]]
ax.barh(cat_sorted["category"], cat_sorted["prob_overconf"],
        color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label="50% (no evidence either way)")
ax.axvline(0.7, color='red',   linestyle=':',  lw=1,   alpha=0.6, label="Strong evidence (>0.7)")
ax.axvline(0.3, color='blue',  linestyle=':',  lw=1,   alpha=0.6, label="Strong evidence (<0.3)")
ax.set_xlabel("Posterior Probability of Overconfidence P(beta < 1)", fontsize=11)
ax.set_title("Which Market Categories Are Overconfident?\n(beta < 1 = market makes too-extreme predictions)", fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig("outputs/figures/18_overconfidence_probability.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 70)
print("POSTERIOR INTERPRETATION SUMMARY")
print("=" * 70)
for _, row in cat_df.sort_values("prob_overconf", ascending=False).iterrows():
    bias = "underestimates YES" if row["alpha_mean"] > 0 else "overestimates YES"
    conf = "overconfident" if row["prob_overconf"] > 0.6 else "underconfident" if row["prob_overconf"] < 0.4 else "well-calibrated slope"
    sig  = "(statistically significant)" if row["alpha_sig"] or row["beta_sig"] else "(not significant)"
    print(f"\n{row['category'].upper()} (n={row['n_markets']}):")
    print(f"  alpha={row['alpha_mean']:+.3f}: Market {bias}")
    print(f"  beta={row['beta_mean']:.3f}: Market is {conf} {sig}")
    print(f"  P(overconfident) = {row['prob_overconf']:.3f}")